In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# install torchmetrics (for mAP)
!pip install -q torchmetrics pycocotools

In [ ]:
import os, sys
import time
import shutil
import random
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import PIL.Image
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import tv_tensors
from torchvision.transforms import v2 as T
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead
from torchmetrics.detection.mean_ap import MeanAveragePrecision

from IPython.display import display, Image

In [ ]:
print(f'torch {torch.__version__}  torchvision {torchvision.__version__}')
print(f'CUDA: {torch.cuda.is_available()} GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

In [ ]:
# CONFIG

# use gpu if available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# don't error on really big source images
PIL.Image.MAX_IMAGE_PIXELS = None

# colab paths: dataset on drive, scratch space on /content
HOME = '/content'
DATA_ROOT = '/content/drive/MyDrive/Dataset'

# fixed seed so we can reproduce the same random samples next time
SEED = 42
# what counts as an image file
IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}

# dataset splits
EXPERIMENTS = ['EXP1_ALL', 'EXP2_OCCLUSION', 'EXP3_CLUSTER', 'EXP4_ILLUMINATION', 'EXP5_SCALE']
MODEL_SIZES = ['SSD300']

# defaults from torchvision ssd reference
# https://github.com/pytorch/vision/blob/main/references/detection/presets.py
# https://docs.pytorch.org/vision/main/models/generated/torchvision.models.detection.ssd300_vgg16.html
# https://github.com/pytorch/vision/tree/main/references/detection

REF_TOTAL_BATCH = 32
REF_LR = 0.002

DEFAULT_WORKERS = min(8, max(2, (os.cpu_count() or 2) - 1))

# type of run (for output dir)
TYPE_OF_RUN = 'SSD'

CONFIG = {
  'epochs': 100,
  'batch_size': 16,
  'image_size': 300,
  # source images are huge: to speed it up we downsample on load so cpu augmentation isn't bottleneck
  'work_max_side': 640,
  'momentum': 0.9,
  'weight_decay': 5e-4,
  'num_workers': DEFAULT_WORKERS,
  'save_path': f'{HOME}/results',
  'warmup_iters': 1000, # ramp lr up over the first 1000 iters so training doesn't diverge early
  'warmup_factor': 1/1000, # start at lr/1000 and grow linearly
}

# scale lr with batch size
CONFIG['lr'] = REF_LR * CONFIG['batch_size'] / REF_TOTAL_BATCH

print(f"Config: lr={CONFIG['lr']:.4f} batch={CONFIG['batch_size']} epochs={CONFIG['epochs']} workers={CONFIG['num_workers']}")

In [ ]:
# how many image files live in a given split folder?
def count_images(split_dir):
  p = Path(split_dir)
  # if the folder doesn't exist, just say 0 instead of crashing
  if not p.exists():
    return 0
  # count files with image extensions
  return sum(1 for f in p.iterdir() if f.suffix.lower() in IMAGE_EXTS)

# one line per object in yolo label files
def count_objects(label_dir):
  p = Path(label_dir)
  if not p.exists():
    return 0
  total = 0
  # add up object counts across all label files
  for lbl in p.glob('*.txt'):
    with open(lbl) as f:
      # ignore blank lines
      total += sum(1 for line in f if line.strip())
  return total

# bundle up image + object counts for all 3 splits into one dict
def get_dataset_sizes(dataset_dir):
  sizes = {}
  # go through each split and count its images + boxes
  for split in ['train', 'valid', 'test']:
    sizes[f'{split}_images'] = count_images(Path(dataset_dir) / split / 'images')
    sizes[f'{split}_oranges'] = count_objects(Path(dataset_dir) / split / 'labels')
  # roll up totals so we can easily see the numbers
  sizes['total_images'] = sum(sizes[f'{s}_images'] for s in ['train', 'valid', 'test'])
  sizes['total_oranges'] = sum(sizes[f'{s}_oranges'] for s in ['train', 'valid', 'test'])
  return sizes

# how many objects are in a single label file?
def ground_truth_count(label_path):
  if not os.path.exists(label_path):
    return 0
  with open(label_path) as f:
    return sum(1 for line in f if line.strip())

In [ ]:
# imagenet-style mean (used to pad blank space when zooming out)
SSD_MEAN_FILL = [123, 117, 104]

# this is the standard ssd augmentation pipeline (random color/zoom/crop/flip)
train_transforms = T.Compose([
  T.RandomPhotometricDistort(), # randomize brightness/contrast/saturation/hue
  T.RandomZoomOut(fill=SSD_MEAN_FILL), # zoom out so the model sees small objects too
  T.RandomIoUCrop(), # crop to a random patch that still contains some boxes
  T.RandomHorizontalFlip(p=0.5), # 50% chance to flip left-right
  T.ToDtype(torch.float, scale=True), # convert [0,255] -> float [0,1]
  T.SanitizeBoundingBoxes(), # drop boxes that got cropped to nothing
])

# val/test: no augmentation, just convert to float
eval_transforms = T.Compose([
  T.ToDtype(torch.float, scale=True),
])

# pytorch dataset that loads our orange images + yolo-style box labels
class OrangeDataset(Dataset):
  def __init__(self, img_dir, label_dir, train=False):
    # remember where images and labels are
    self.img_dir = Path(img_dir)
    self.label_dir = Path(label_dir)
    # get all image paths once at start (sorted for determinism)
    self.image_paths = sorted(p for p in self.img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    # train mode = augmentations on, eval mode = augmentations off
    self.transforms = train_transforms if train else eval_transforms

  # pytorch needs this to know the dataset size
  def __len__(self):
    return len(self.image_paths)

  # load one image + its boxes, run transforms, return back tensors
  def __getitem__(self, idx):
    # pull file path for this index
    img_path = self.image_paths[idx]
    # open the image and force rgb
    with PIL.Image.open(img_path) as pim:
      pim = pim.convert('RGB')
      max_side = CONFIG.get('work_max_side', 0)
      # shrink huge images here, boxes are normalized so they're fine
      if max_side and max(pim.size) > max_side:
        scale = max_side / max(pim.size)
        new_size = (int(round(pim.size[0] * scale)), int(round(pim.size[1] * scale)))
        pim = pim.resize(new_size, PIL.Image.BILINEAR)
      # copy=True gives us a writable array
      img_np = np.array(pim, copy=True)
    H, W = img_np.shape[:2]
    # numpy is HWC, pytorch wants CHW, so permute
    # wrap in tv_tensor so transforms know it's an image
    img_t = tv_tensors.Image(torch.from_numpy(img_np).permute(2, 0, 1).contiguous())

    # parse the label file into pixel-space xyxy boxes
    boxes, labels = [], []
    # label file has the same stem as the image (e.g. foo.jpg -> foo.txt)
    label_path = self.label_dir / f'{img_path.stem}.txt'
    if label_path.exists():
      # yolo label format: class cx cy w h, all normalized 0-1
      for line in open(label_path):
        parts = line.split()
        if len(parts) != 5:
          continue
        # _ is the class id (we already know it's orange so ignore)
        _, cx, cy, bw, bh = map(float, parts)
        # convert from normalized cx/cy/w/h to absolute xyxy in pixel coords
        # also clamp to image bounds in case a box bleeds over the edge
        x1 = max(0.0, (cx - bw / 2) * W)
        y1 = max(0.0, (cy - bh / 2) * H)
        x2 = min(float(W), (cx + bw / 2) * W)
        y2 = min(float(H), (cy + bh / 2) * H)
        # drop degenerate boxes
        if x2 > x1 + 1 and y2 > y1 + 1:
          boxes.append([x1, y1, x2, y2])
          labels.append(1)

    # to tensors
    # need the empty case so images with no boxes don't crash the model
    boxes_t = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)

    # wrap boxes in tv_tensors so the v2 transforms know to update them with the image
    target = {
      'boxes': tv_tensors.BoundingBoxes(boxes_t, format='XYXY', canvas_size=(H, W)),
      'labels': labels_t,
    }
    # apply augmentations (or just float-convert in eval mode)
    img_out, target = self.transforms(img_t, target)
    # unwrap back to plain tensors (model doesn't want tv_tensor types)
    target = {'boxes': torch.as_tensor(target['boxes'], dtype=torch.float32),
              'labels': target['labels']}
    # also return path so we can identify which
    return img_out, target, str(img_path)

# detection batches are ragged (different num boxes per image) so we just keep them as lists
def collate_fn(batch):
  # unpack the (image, target, path) tuples into 3 separate lists
  images, targets, paths = zip(*batch)
  return list(images), list(targets), list(paths)

# load pretrained ssd300 and adapt it to our class count
def build_ssd_model(num_classes=2):
  # start from coco-pretrained weights so we don't have to learn features from scratch
  model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)
  # ssd300 has 6 detection feature maps (these are their channel counts)
  in_channels = [512, 1024, 512, 256, 256, 256]
  # how many anchor boxes are placed at each spatial location of each feature map
  num_anchors = model.anchor_generator.num_anchors_per_location()
  # swap classification head for our class count (bg + orange)
  model.head.classification_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes,
  )
  # move to device
  return model.to(DEVICE)

In [ ]:
# train one ssd run end-to-end and save best and last checkpoints
def run_model(dataset, model_size, exp_name, trace=True, val_every=1):
  # set up output paths so each experiment gets its own folder
  run_name = f'ssd300_{exp_name}'
  run_dir = os.path.join(CONFIG['save_path'], run_name)
  weights_dir = os.path.join(run_dir, 'weights')
  os.makedirs(weights_dir, exist_ok=True)
  best_path = os.path.join(weights_dir, 'best.pt') # best = lowest val loss seen so far
  last_path = os.path.join(weights_dir, 'last.pt') # last = whatever the latest epoch saved

  # build train and val datasets (train=True turns on augmentations!)
  train_ds = OrangeDataset(Path(dataset.location)/'train'/'images', Path(dataset.location)/'train'/'labels', train=True)
  val_ds = OrangeDataset(Path(dataset.location)/'valid'/'images', Path(dataset.location)/'valid'/'labels', train=False)

  # dataloader handles batching + parallel loading, shuffle=True for train so each epoch is different
  # pin_memory + persistent_workers to speed up gpu transfers and avoid worker spin-up each epoch
  train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn, num_workers=CONFIG['num_workers'],
                            pin_memory=True, persistent_workers=CONFIG['num_workers'] > 0)
  # val loader doesn't shuffle - we want consistent order so val loss is comparable across epochs
  val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=CONFIG['num_workers'],
                          pin_memory=True, persistent_workers=CONFIG['num_workers'] > 0)

  # build fresh model for this run (don't reuse weights between experiments)
  model = build_ssd_model(num_classes=2)
  optimizer = optim.SGD(model.parameters(), lr=CONFIG['lr'], momentum=CONFIG['momentum'], weight_decay=CONFIG['weight_decay'])

  # linear warmup, then cosine for the rest
  warmup_iters = min(CONFIG['warmup_iters'], len(train_loader) * CONFIG['epochs'] - 1)
  warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=CONFIG['warmup_factor'], total_iters=warmup_iters)
  step_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

  # track best val loss so we know when to overwrite best.pt
  best_val_loss = float('inf')
  train_losses, val_losses = [], []
  global_iter = 0
  skipped_batches = 0

  # main training loop
  for epoch in range(CONFIG['epochs']):
    epoch_start = time.time()

    model.train()
    epoch_loss = 0.0
    n_batches = 0

    # one pass over training set
    for images, targets, _ in train_loader:
      # ship batch over to gpu (non_blocking lets data load while gpu is busy)
      images = [img.to(DEVICE, non_blocking=True) for img in images]
      targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]

      # bf16 amp for speed boost without messing with grad scaling
      with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())

      # skip nan/inf batches so they don't blow up training
      if not torch.isfinite(losses):
        optimizer.zero_grad()
        skipped_batches += 1
        global_iter += 1
        if global_iter < warmup_iters:
          warmup_scheduler.step()
        continue

      # clear old grads, backprop, update weights
      optimizer.zero_grad()
      losses.backward()
      optimizer.step()

      if global_iter < warmup_iters:
        warmup_scheduler.step()
      global_iter += 1

      # accumulate so we can print average train loss at end of epoch
      epoch_loss += losses.item()
      n_batches += 1

    # mean train loss for this epoch
    avg_train_loss = epoch_loss / max(n_batches, 1)
    train_losses.append(avg_train_loss)

    # only run val every few epochs to save time
    # original ssd does every 10000 iterations, while yolo does every epoch
    # so we also do every epoch
    is_val_epoch = ((epoch + 1) % val_every == 0) or (epoch == CONFIG['epochs'] - 1)
    if is_val_epoch:
      # torchvision detection only returns loss in train mode
      model.train()
      val_loss = 0.0
      with torch.no_grad():
        for images, targets, _ in val_loader:
          images = [img.to(DEVICE, non_blocking=True) for img in images]
          targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]
          with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            loss_dict = model(images, targets)
            val_loss += sum(loss_dict.values()).item()
      avg_val_loss = val_loss / max(len(val_loader), 1)
      val_losses.append(avg_val_loss)

      # if this is a new best, save it as best.pt
      if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), best_path)
    else:
      # not a val epoch - just repeat the last value so the plot stays the right length
      val_losses.append(val_losses[-1] if val_losses else float('nan'))

    # cosine scheduler only ticks after warmup is done, otherwise the lr swings around
    if global_iter >= warmup_iters:
      step_scheduler.step()

    # save last
    torch.save(model.state_dict(), last_path)

    elapsed = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']
    val_str = f'val={val_losses[-1]:.4f}' if is_val_epoch else 'val= - '
    print(f'[{epoch+1:3d}/{CONFIG["epochs"]}] train={avg_train_loss:.4f} {val_str} best={best_val_loss:.4f} lr={current_lr:.2e} time={elapsed:.1f}s', flush=True)

  print(f'\nBest val loss: {best_val_loss:.4f} -> {best_path}')
  if skipped_batches > 0:
    print(f'Skipped {skipped_batches} non-finite batches during training')

  # sanity-check
  if trace:
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='train')
    plt.plot(val_losses, label='val')
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title(f'SSD300 {exp_name}')
    plt.tight_layout()
    curve_path = os.path.join(run_dir, 'loss_curves.png')
    plt.savefig(curve_path)
    plt.show()

  # copy results to drive
  drive_run_dir = os.path.join(DATA_ROOT, 'saved_models', TYPE_OF_RUN, run_name)
  shutil.copytree(run_dir, drive_run_dir, dirs_exist_ok=True)
  print(f'Saved run to: {drive_run_dir}')

  return model, None, run_dir, best_path

In [ ]:
@torch.no_grad()
# run model on a single image and return whatever survives the conf threshold
def _predict_image(model, img_path, conf, image_size):
  # load image and convert to rgb numpy array
  with PIL.Image.open(img_path) as pim:
    img_np = np.asarray(pim.convert('RGB'))
  # ssd handles its own resize internally and remaps boxes back
  # HWC uint8 -> CHW float [0,1] -> add batch dim -> gpu
  t = torch.from_numpy(img_np).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(DEVICE)
  # [0] because we passed in a batch of 1
  out = model(t)[0]
  scores = out['scores'].cpu()
  # mask of detections that pass the confidence threshold
  keep = scores >= conf
  boxes = out['boxes'][keep].cpu()
  return {'boxes': boxes, 'scores': scores[keep], 'labels': out['labels'][keep].cpu(), 'count': int(keep.sum())}

# sweep a bunch of confidence thresholds on the val set and pick the one with lowest count error
def search_conf_threshold(best_path, dataset_location,
                          thresholds=(0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9)):
  # rebuild the model architecture and load best weights
  model = build_ssd_model(num_classes=2)
  model.load_state_dict(torch.load(best_path, map_location=DEVICE))
  model.eval() # eval mode = no dropout, return predictions instead of loss

  # we tune on val set
  val_img_dir = Path(dataset_location) / 'valid' / 'images'
  val_label_dir = Path(dataset_location) / 'valid' / 'labels'
  image_paths = sorted(p for p in val_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

  # try each threshold, see which gives the closest predicted count to ground truth
  results = []
  for conf in thresholds:
    errors = []
    # for every val image, get gt count and predicted count at this threshold
    for img_path in image_paths:
      gt = ground_truth_count(val_label_dir / f'{img_path.stem}.txt')
      pred = _predict_image(model, img_path, conf, CONFIG['image_size'])
      errors.append(pred['count'] - gt)
    # mae = mean absolute error, rmse = root mean squared error (penalizes big mistakes more)
    errors = np.array(errors, dtype=float)
    mae = float(np.mean(np.abs(errors)))
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    results.append({'conf': conf, 'mae': mae, 'rmse': rmse})
    print(f'conf={conf:.2f} MAE={mae:.3f} RMSE={rmse:.3f}')

  # find threshold with lowest mae
  df = pd.DataFrame(results)
  best = df.loc[df['mae'].idxmin()]
  print(f"\nBest threshold: {best['conf']} (MAE={best['mae']:.3f})")
  return df, float(best['conf'])

# run model on test set and compute metrics
def eval_model(best_path, dataset, conf=0.25):
  model = build_ssd_model(num_classes=2)
  model.load_state_dict(torch.load(best_path, map_location=DEVICE))
  model.eval()

  # build a test loader (no shuffle, no augmentation)
  test_ds = OrangeDataset(Path(dataset.location)/'test'/'images', Path(dataset.location)/'test'/'labels', train=False)
  test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=CONFIG['num_workers'])

  # mAP from torchmetrics
  # max detections = 500 because images can have lots of oranges (but for us we have < 500 so this is just upper threshold)
  metric = MeanAveragePrecision(iou_type='bbox', max_detection_thresholds=[1, 10, 500])

  gt_counts, pred_counts = [], []
  # tracking these for precision/recall at iou>=0.5
  tp = fp = fn = 0
  iou_thr = 0.5

  with torch.no_grad():
    for images, targets, _ in test_loader:
      images = [img.to(DEVICE) for img in images]
      outputs = model(images)
      preds_for_metric, tgts_for_metric = [], []

      # process each image in the batch one at a time
      for output, target in zip(outputs, targets):
        # filter out low-confidence preds before computing metrics
        scores = output['scores'].cpu()
        keep = scores >= conf
        pred_boxes = output['boxes'][keep].cpu()
        gt_boxes = target['boxes']

        preds_for_metric.append({'boxes': pred_boxes, 'scores': scores[keep], 'labels': output['labels'][keep].cpu()})
        tgts_for_metric.append({'boxes': gt_boxes, 'labels': target['labels']})
        pred_counts.append(int(keep.sum()))
        gt_counts.append(len(gt_boxes))

        # greedy match: pair each pred with the first gt that overlaps enough
        matched_gt = set()
        # for each pred, walk through gt boxes and find the first one that overlaps enough
        for pb in pred_boxes:
          matched = False
          for gi, gb in enumerate(gt_boxes):
            # don't double-count - skip gt boxes already matched to another pred
            if gi in matched_gt:
              continue
            # iou math: intersection rect = max of mins, min of maxes
            ix1, iy1 = max(pb[0], gb[0]), max(pb[1], gb[1])
            ix2, iy2 = min(pb[2], gb[2]), min(pb[3], gb[3])
            # max(0, ...) handles case where boxes don't overlap at all
            inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
            # union = sum of areas - intersection (otherwise we would double-count the overlap)
            union = (pb[2]-pb[0])*(pb[3]-pb[1]) + (gb[2]-gb[0])*(gb[3]-gb[1]) - inter
            # if iou is high enough, count it as a true positive and stop searching
            if union > 0 and inter / union >= iou_thr:
              tp += 1; matched_gt.add(gi); matched = True; break
          # no gt matched -> this pred is a false positive
          if not matched:
            fp += 1
        # any gt boxes left unmatched are false negatives (we missed them)
        fn += len(gt_boxes) - len(matched_gt)

      metric.update(preds_for_metric, tgts_for_metric)

  # finalize torchmetrics aggregation
  results = metric.compute()
  # standard precision/recall/f1 - guard against div-by-zero
  p = tp / (tp + fp) if (tp + fp) else 0.0
  r = tp / (tp + fn) if (tp + fn) else 0.0
  f1 = (2 * p * r) / (p + r) if (p + r) else 0.0

  # per-image count error (positive = over-counted, negative = under-counted)
  errors = np.array(pred_counts) - np.array(gt_counts)
  return {
    'box_precision': p,
    'box_recall': r,
    'box_f1': f1,
    'box_map50': float(results['map_50']),
    'box_map': float(results['map']),
    'mae': float(np.mean(np.abs(errors))),
    'rmse': float(np.sqrt(np.mean(errors ** 2))),
  }

In [ ]:
# end-to-end pipeline for one experiment: train, tune conf, evaluate
def run_experiment(exp_name):
  # the bbox-format dataset lives directly on drive
  box_root = os.path.join(DATA_ROOT, exp_name, 'YOLO_BOX')
  # skip cleanly if we don't have the data for this one
  if not os.path.isdir(box_root):
    print(f'\n{exp_name}: dataset not found at {box_root}, skipping.')
    return pd.DataFrame()

  print(f'\n{exp_name}')
  print(f'Dataset: {box_root}')

  # tiny shim so the rest of the code can do dataset.location
  dataset = SimpleNamespace(location=box_root)
  # quick sanity-check sizes so we know we found the right folder
  print(get_dataset_sizes(box_root))

  # we only have ssd300 right now, but loop in case we add more sizes later
  rows = []
  for ms in MODEL_SIZES:
    # time the training so we can report it
    start = datetime.now()
    _, _, run_dir, best_path = run_model(dataset, model_size=ms, exp_name=exp_name)
    end = datetime.now()

    # find the best confidence threshold on val, then evaluate on test with it
    _, best_conf = search_conf_threshold(best_path, box_root)
    metrics = eval_model(best_path, dataset, conf=best_conf)

    # collect everything we want to log into one row of the results table
    rows.append({
      'experiment': exp_name,
      'model': 'SSD',
      'model_size': ms,
      'run_dir': run_dir,
      'best_path': best_path,
      'conf': best_conf,
      'train_time': end - start,
      **metrics,
    })

  return pd.DataFrame(rows)


# clear out gpu memory before the long run
torch.cuda.empty_cache()

# run every experiment back-to-back
all_results = [run_experiment(exp) for exp in EXPERIMENTS]

# drop empty frames (from skipped experiments) and stack the rest into one big table
all_results = [df for df in all_results if not df.empty]
results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

out_csv = f'{DATA_ROOT}/ssd_results-{datetime.now():%Y%m%d-%H%M%S}.csv'
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
results_df.to_csv(out_csv, index=False)
print(f'wrote {out_csv}')
results_df

In [ ]:
# how many test images to draw per experiment
NUM_SAMPLES = 5

# loop through every experiment and visualize a few test predictions
for exp_name in EXPERIMENTS:
  # nothing to show if there are no results at all
  if results_df.empty:
    break
  # might have skipped this experiment, skip it here too
  subset = results_df[results_df['experiment'] == exp_name]
  if subset.empty:
    continue

  # pick best run for this experiment (highest mAP)
  row = subset.loc[subset['box_map'].idxmax()]
  # rebuild + load weights for that run
  model = build_ssd_model(num_classes=2)
  model.load_state_dict(torch.load(row['best_path'], map_location=DEVICE))
  model.eval()
  # use tuned confidence threshold from eval step
  conf = row['conf']

  box_root = os.path.join(DATA_ROOT, exp_name, 'YOLO_BOX')
  test_img_dir = Path(box_root) / 'test' / 'images'
  test_label_dir = Path(box_root) / 'test' / 'labels'
  image_paths = sorted(p for p in test_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

  # fixed seed so same images show up every time we re-run
  random.seed(SEED)
  sample_paths = random.sample(image_paths, min(NUM_SAMPLES, len(image_paths)))

  # one row per sample image
  fig, axes = plt.subplots(len(sample_paths), 1, figsize=(14, 6 * len(sample_paths)))
  if len(sample_paths) == 1:
    axes = [axes]

  for i, img_path in enumerate(sample_paths):
    with PIL.Image.open(img_path) as pim:
      img_rgb = np.asarray(pim.convert('RGB'))
    label_path = test_label_dir / f'{img_path.stem}.txt'
    gt_count = ground_truth_count(label_path)

    # run model on this image
    pred = _predict_image(model, img_path, conf, CONFIG['image_size'])
    axes[i].imshow(img_rgb)
    # draw rectangle for every kept detection
    for box in pred['boxes'].numpy():
      x1, y1, x2, y2 = map(int, box)
      # bright green boxes pop nicely against orange fruit
      axes[i].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                      fill=False, edgecolor=(105/255, 255/255, 99/255), linewidth=2))
    axes[i].set_title(f'pred: {pred["count"]} / gt: {gt_count} (conf>={conf:.2f}, size={row["model_size"]})',
                      fontsize=14, fontweight='bold')
    axes[i].axis('off')

  plt.suptitle(f'Detection: {exp_name}', fontsize=16, fontweight='bold', y=1.0)
  plt.tight_layout()
  plt.show()